# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive, step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema file URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
md = dataset.metadata
print(f"Dataset: {md.name}")
print(f"Version: {md.version}")
print(f"Description: {md.description}\n")
print(f"Authors: {md.author}")
print(f"Identifier: {md.identifier}")
print(f"Published: {md.datePublished}")
print(f"Keywords: {md.keywords}")

## 2. Data Overview
Explore the available record sets, fields, and their unique `@id`s as defined in the Croissant schema.

The `mlcroissant` API gives you access to record set definitions via their `@id`. 

In [ ]:
# List all record sets and their @ids
print("Available Record Sets and their @ids:")
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"  - Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {rs.description}\n")

# List fields in each record set
for rs in record_sets:
    print(f"Fields for record set '{rs.name}':")
    for field in rs.fields:
        print(f"  - Field: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Data type: {field.data_type}")
        print(f"      Description: {field.description}")
        if hasattr(field, 'column') and field.column:
            print(f"      Column(s): {field.column}")
    print()

## 3. Data Extraction
Here, we load data from a selected record set into a pandas DataFrame for further analysis.

You must reference record sets and fields by their `@id` as shown above.

In [ ]:
# For demonstration, pick the first available record set
record_set_ids = [rs.id for rs in dataset.record_sets]
# You may adjust this to target a specific record set if desired
print(f"Chosen record set for extraction: {record_set_ids[0]}")

dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records for record set @id: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows with columns: {list(df.columns)}\n")
    else:
        print("  (No records found)\n")

# We'll use the first loaded DataFrame for further exploration
main_rs_id = record_set_ids[0]

df = dataframes.get(main_rs_id, pd.DataFrame())
print("Preview of the extracted DataFrame:")
df.head()

## 4. Exploratory Data Analysis (EDA)
We will demonstrate operations such as filtering, normalization, and grouping on relevant numeric fields. Make sure to use field `@id`s for column references.

**Note:** Adjust the field `@id` names in code below according to fields you discover in the overview for your specific analysis goal.

In [ ]:
# Identify a likely numeric field for filtering - modify as needed!
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use the first numeric column found
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = None
    print("No numeric fields found!")

if numeric_field_id:
    # Set an example threshold or derive one from the data
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize this field
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by a categorical field, if available
    group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].agg(['mean','count'])
        print(f"Grouped and averaged by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize distributions or relationships between fields. You can adjust columns according to your needs and the schema overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot example if there are at least two numeric fields
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(
            x=df[numeric_fields[0]],
            y=df[numeric_fields[1]]
        )
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f'{numeric_fields[0]} vs {numeric_fields[1]}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset via its Croissant schema and explored its content using the `mlcroissant` library. We demonstrated how to access record sets and fields by their `@id`s, loaded the data into DataFrames, performed basic EDA with filtering and normalization, and showed example visualizations of numeric attributes.

For in-depth analysis, repeat these patterns using the field and record set `@id`s relevant for your scientific investigation.